# PostgreSQL: безопасное чтение источника


Показать обнаружение схемы, параметры, выбор полей, агрегацию и профилирование при отсутствии секретов.

Нужен пакет `psycopg2` и переменная `SOURCE_POSTGRES_DSN`. Без неё каждая ячейка сообщает о пропуске; DSN никогда не печатается.

Ограничения: Для точечной диагностики внешнего PostgreSQL перед регулярной загрузкой. Не вставляйте значения в SQL строковой подстановкой; используйте отдельную учётную запись только для чтения.


## Основные понятия
Соединение создаётся только из `SOURCE_POSTGRES_DSN`, переводится в режим только чтения и закрывается в той же ячейке.


In [ ]:
import os
import psycopg2

## Примеры


**1. Доступность**

Проверьте наличие DSN, не печатая его значение.



In [ ]:
dsn = os.getenv("SOURCE_POSTGRES_DSN")
print("PostgreSQL настроен:", bool(dsn))

**2. Схема**

Получите имена таблиц из системного каталога с ограничением выдачи.



In [ ]:
if not os.getenv("SOURCE_POSTGRES_DSN"):
    print("Пример пропущен: SOURCE_POSTGRES_DSN не задан")
else:
    with psycopg2.connect(os.environ["SOURCE_POSTGRES_DSN"]) as connection:
        connection.set_session(readonly=True, autocommit=True)
        with connection.cursor() as cursor:
            cursor.execute("select table_schema, table_name from information_schema.tables where table_schema not in ('pg_catalog', 'information_schema') order by table_schema, table_name limit 30")
            print(*cursor.fetchall(), sep="\n")


**3. Параметризованный выбор**

Передайте значения отдельно от SQL и ограничьте строки.



In [ ]:
if not os.getenv("SOURCE_POSTGRES_DSN"):
    print("Пример пропущен: SOURCE_POSTGRES_DSN не задан")
else:
    with psycopg2.connect(os.environ["SOURCE_POSTGRES_DSN"]) as connection:
        connection.set_session(readonly=True, autocommit=True)
        with connection.cursor() as cursor:
            cursor.execute("select match_id, duration from dota.matches where duration >= %s order by match_id desc limit %s", (1800, 10))
            print(*cursor.fetchall(), sep="\n")


**4. Агрегация**

Выполните небольшую агрегацию на стороне PostgreSQL.



In [ ]:
if not os.getenv("SOURCE_POSTGRES_DSN"):
    print("Пример пропущен: SOURCE_POSTGRES_DSN не задан")
else:
    with psycopg2.connect(os.environ["SOURCE_POSTGRES_DSN"]) as connection:
        connection.set_session(readonly=True, autocommit=True)
        with connection.cursor() as cursor:
            cursor.execute("select radiant_win, count(*) from dota.matches group by radiant_win order by radiant_win")
            print(*cursor.fetchall(), sep="\n")


**5. Профиль полей**

Получите типы и несколько значений, не выгружая всю таблицу.



In [ ]:
if not os.getenv("SOURCE_POSTGRES_DSN"):
    print("Пример пропущен: SOURCE_POSTGRES_DSN не задан")
else:
    with psycopg2.connect(os.environ["SOURCE_POSTGRES_DSN"]) as connection:
        connection.set_session(readonly=True, autocommit=True)
        with connection.cursor() as cursor:
            cursor.execute("select column_name, data_type from information_schema.columns where table_schema = 'dota' and table_name = 'matches' order by ordinal_position")
            print(*cursor.fetchall(), sep="\n")
